# ARIMA Model Optimization for Sales Prediction
## Article 94869 - L'Oreal Product
### Optimized parameter tuning and comprehensive model evaluation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from pmdarima import auto_arima
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Prepare Data

In [2]:
# Load data
df = pd.read_csv("../data_raw/date_24_art_loreal_20%.csv")

# Filter for article 94869
df = df[df["ID_ARTICOL"] == 94869].copy()

# Convert DATA column to datetime
df['DATA'] = pd.to_datetime(df['DATA'], errors="coerce")

# Sort by date
df = df.sort_values(by=["DATA"]).reset_index(drop=True)

print(f"Data shape: {df.shape}")
print(f"Date range: {df['DATA'].min()} to {df['DATA'].max()}")
print(f"\nFirst few rows:")
print(df.head())

Data shape: (733, 13)
Date range: 2024-01-01 00:00:00 to 2026-01-04 00:00:00

First few rows:
        DATA  ID_ARTICOL                                            ARTICOL  \
0 2024-01-01       94869  CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...   
1 2024-01-02       94869  CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...   
2 2024-01-03       94869  CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...   
3 2024-01-04       94869  CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...   
4 2024-01-05       94869  CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...   

   CANTITATE  VAL_IESIRE_FARA_TVA  VAL_IESIRE_CU_TVA  VAL_INTRARE_FARA_TVA  \
0          0                 0.00                0.0                  0.00   
1          0                 0.00                0.0                  0.00   
2         10               571.46              680.0                508.30   
3          4               228.57              272.0                203.32   
4         12               685.69        

In [3]:
# Extract time series for CANTITATE
ts = df.set_index('DATA')['CANTITATE'].astype(float).sort_index()

# Check for missing values
print(f"Missing values: {ts.isna().sum()}")
print(f"Time series length: {len(ts)}")

# Create train-test split (80-20)
split_ratio = None  # using last 30 days as test
split_idx = len(ts) - 30
train = ts.iloc[:split_idx]
test = ts.iloc[split_idx:]

print(f"\nTrain set: {len(train)} observations")
print(f"Test set: {len(test)} observations")

Missing values: 0
Time series length: 733

Train set: 586 observations
Test set: 147 observations


## 2. Time Series Visualization

In [5]:
fig = px.line(
    x=ts.index, y=ts.values,
    title='Time Series: CANTITATE (Article 94869)',
    labels={'x': 'Data', 'y': 'Cantitate'},
    markers=False
)

fig.add_vline(x=train.index[-1], line_dash="dash", line_color="red", annotation_text="Train/Test Split")

fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=list([
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(count=3, label="3m", step="month", stepmode="backward"),
            dict(count=6, label="6m", step="month", stepmode="backward"),
            dict(count=1, label="1y", step="year", stepmode="backward"),
            dict(step="all")
        ])
    )
)

fig.update_layout(height=500, hovermode='x unified')
fig.show()

TypeError: Addition/subtraction of integers and integer-arrays with Timestamp is no longer supported.  Instead of adding/subtracting `n`, use `n * obj.freq`

## 3. Stationarity Testing (ADF & KPSS)

In [ ]:
def test_stationarity(timeseries, name=""):
    """Test for stationarity using ADF and KPSS tests"""
    print(f"\n{'='*60}")
    print(f"Stationarity Tests for {name}")
    print(f"{'='*60}")
    
    # ADF Test
    adf_result = adfuller(timeseries.dropna())
    print(f"\nADF Test (Null hypothesis: Series is non-stationary)")
    print(f"  ADF Statistic: {adf_result[0]:.6f}")
    print(f"  p-value: {adf_result[1]:.6f}")
    print(f"  Critical Values:")
    for key, value in adf_result[4].items():
        print(f"    {key}: {value:.3f}")
    
    if adf_result[1] <= 0.05:
        print(f"  ✓ Series is STATIONARY (reject null hypothesis)")
        adf_stationary = True
    else:
        print(f"  ✗ Series is NON-STATIONARY (fail to reject null hypothesis)")
        adf_stationary = False
    
    # KPSS Test
    kpss_result = kpss(timeseries.dropna(), regression='c', nlags='auto')
    print(f"\nKPSS Test (Null hypothesis: Series is stationary)")
    print(f"  KPSS Statistic: {kpss_result[0]:.6f}")
    print(f"  p-value: {kpss_result[1]:.6f}")
    print(f"  Critical Values:")
    for key, value in kpss_result[3].items():
        print(f"    {key}: {value:.3f}")
    
    if kpss_result[1] >= 0.05:
        print(f"  ✓ Series is STATIONARY (fail to reject null hypothesis)")
        kpss_stationary = True
    else:
        print(f"  ✗ Series is NON-STATIONARY (reject null hypothesis)")
        kpss_stationary = False
    
    print(f"\nConclusion: Series is {'STATIONARY' if (adf_stationary and kpss_stationary) else 'NON-STATIONARY'}")
    
    return adf_stationary and kpss_stationary

# Test original series
is_stationary = test_stationarity(train, "Original Training Series")

In [ ]:
# Test differenced series if not stationary
if not is_stationary:
    train_diff = train.diff().dropna()
    is_diff_stationary = test_stationarity(train_diff, "First Differenced Series")
    
    if not is_diff_stationary:
        train_diff2 = train_diff.diff().dropna()
        is_diff2_stationary = test_stationarity(train_diff2, "Second Differenced Series")
        d_order = 2
    else:
        d_order = 1
else:
    d_order = 0

print(f"\n{'='*60}")
print(f"Recommended differencing order (d): {d_order}")
print(f"{'='*60}")

## 4. ACF and PACF Plots for Parameter Identification

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# ACF and PACF for original series
plot_acf(train.dropna(), lags=30, ax=axes[0, 0], title='ACF - Original Series')
plot_pacf(train.dropna(), lags=30, ax=axes[0, 1], title='PACF - Original Series')

# ACF and PACF for differenced series
if d_order > 0:
    train_diff = train.diff().dropna()
    plot_acf(train_diff, lags=30, ax=axes[1, 0], title=f'ACF - {d_order}st Differenced')
    plot_pacf(train_diff, lags=30, ax=axes[1, 1], title=f'PACF - {d_order}st Differenced')
else:
    axes[1, 0].text(0.5, 0.5, 'No differencing needed', ha='center', va='center')
    axes[1, 1].text(0.5, 0.5, 'Series is already stationary', ha='center', va='center')
    axes[1, 0].set_xticks([])
    axes[1, 0].set_yticks([])
    axes[1, 1].set_xticks([])
    axes[1, 1].set_yticks([])

plt.tight_layout()
plt.show()

print("\nACF/PACF Analysis:")
print("- ACF (AutoCorrelation Function): Shows correlation with past values")
print("- PACF (Partial AutoCorrelation Function): Shows direct correlation after removing intermediate lags")
print("- Use these plots to guide p and q parameter selection")

## 5. Auto ARIMA - Automatic Parameter Selection

In [ ]:
print("Running Auto ARIMA to find optimal parameters...")
print("This may take a minute...\n")

auto_model = auto_arima(
    train,
    seasonal=False,
    d=d_order,
    max_p=6,
    max_q=6,
    max_d=2,
    stepwise=True,
    suppress_warnings=True,
    trace=False,
    error_action='ignore'
)

auto_order = auto_model.order
print(f"\n{'='*60}")
print(f"Auto ARIMA Optimal Parameters: {auto_order}")
print(f"AIC: {auto_model.aic:.2f}")
print(f"BIC: {auto_model.bic:.2f}")
print(f"{'='*60}")
print(f"\nModel Summary:\n{auto_model.summary()}")

## 6. Manual Grid Search - Test Multiple ARIMA Configurations

In [ ]:
def evaluate_arima(order, train_data, test_data):
    """Fit ARIMA model and return evaluation metrics"""
    try:
        model = ARIMA(train_data, order=order)
        fitted = model.fit()
        
        # Forecast
        predictions = fitted.forecast(steps=len(test_data))
        
        # Calculate metrics
        mae = mean_absolute_error(test_data, predictions)
        rmse = np.sqrt(mean_squared_error(test_data, predictions))
        mape = mean_absolute_percentage_error(test_data, predictions)
        r2 = r2_score(test_data, predictions)
        
        # Correlation between actual and predicted
        corr = np.corrcoef(test_data, predictions)[0, 1]
        
        return {
            'Order': order,
            'AIC': fitted.aic,
            'BIC': fitted.bic,
            'MAE': mae,
            'RMSE': rmse,
            'MAPE': mape,
            'R2': r2,
            'Correlation': corr,
            'Model': fitted
        }
    except Exception as e:
        return None

# Grid search for p, d, q
print("Performing grid search on ARIMA parameters...")
print("Testing combinations of p (0-6), d (0-2), q (0-6)\n")

results_list = []
p_range = range(0, 7)
d_range = range(0, 3)
q_range = range(0, 7)

for p in p_range:
    for d in d_range:
        for q in q_range:
            order = (p, d, q)
            result = evaluate_arima(order, train, test)
            if result is not None:
                results_list.append(result)

# Create results dataframe
results_df = pd.DataFrame([
    {
        'Order': r['Order'],
        'AIC': r['AIC'],
        'BIC': r['BIC'],
        'MAE': r['MAE'],
        'RMSE': r['RMSE'],
        'MAPE': r['MAPE'],
        'R2': r['R2'],
        'Correlation': r['Correlation']
    } for r in results_list
])

print(f"\nTested {len(results_df)} configurations")
print(f"\nTop 10 models by AIC:")
print(results_df.nsmallest(10, 'AIC')[['Order', 'AIC', 'BIC', 'MAE', 'RMSE', 'R2']])

In [ ]:
print(f"\nTop 10 models by RMSE:")
print(results_df.nsmallest(10, 'RMSE')[['Order', 'AIC', 'MAE', 'RMSE', 'MAPE', 'R2']])

print(f"\nTop 10 models by R²:")
print(results_df.nlargest(10, 'R2')[['Order', 'AIC', 'MAE', 'RMSE', 'MAPE', 'R2']])

## 7. Select Best Model by Multiple Criteria

In [ ]:
# Rank models by different criteria
results_df['AIC_Rank'] = results_df['AIC'].rank()
results_df['RMSE_Rank'] = results_df['RMSE'].rank()
results_df['R2_Rank'] = results_df['R2'].rank(ascending=False)
results_df['Avg_Rank'] = (results_df['AIC_Rank'] + results_df['RMSE_Rank'] + results_df['R2_Rank']) / 3

# Get best models
best_aic = results_df.nsmallest(1, 'AIC').iloc[0]
best_rmse = results_df.nsmallest(1, 'RMSE').iloc[0]
best_r2 = results_df.nlargest(1, 'R2').iloc[0]
best_overall = results_df.nsmallest(1, 'Avg_Rank').iloc[0]

print("\n" + "="*70)
print("BEST MODELS BY DIFFERENT CRITERIA")
print("="*70)

print(f"\n1. Best by AIC (Akaike Information Criterion):")
print(f"   Order: {best_aic['Order']}, AIC: {best_aic['AIC']:.2f}, MAE: {best_aic['MAE']:.4f}, RMSE: {best_aic['RMSE']:.4f}")

print(f"\n2. Best by RMSE (Root Mean Squared Error):")
print(f"   Order: {best_rmse['Order']}, RMSE: {best_rmse['RMSE']:.4f}, AIC: {best_rmse['AIC']:.2f}, MAE: {best_rmse['MAE']:.4f}")

print(f"\n3. Best by R² (Coefficient of Determination):")
print(f"   Order: {best_r2['Order']}, R²: {best_r2['R2']:.4f}, RMSE: {best_r2['RMSE']:.4f}, MAE: {best_r2['MAE']:.4f}")

print(f"\n4. Best Overall (Average Ranking):")
print(f"   Order: {best_overall['Order']}, Avg_Rank: {best_overall['Avg_Rank']:.2f}")
print(f"   AIC: {best_overall['AIC']:.2f}, RMSE: {best_overall['RMSE']:.4f}, R²: {best_overall['R2']:.4f}")

print("\n" + "="*70)
print("SELECTED MODEL: Best Overall by Average Ranking")
print("="*70)

selected_order = tuple(best_overall['Order'])
print(f"Order: {selected_order}")

## 8. Fit Best Model and Generate Predictions

In [ ]:
# Fit the best model
best_model = ARIMA(train, order=selected_order)
best_fitted = best_model.fit()

# Generate predictions on test set
predictions = best_fitted.forecast(steps=len(test))

print(f"Model Summary for Order {selected_order}:")
print(best_fitted.summary())

## 9. Comprehensive Model Evaluation Metrics

In [ ]:
def calculate_all_metrics(actual, predicted):
    """Calculate comprehensive evaluation metrics"""
    mae = mean_absolute_error(actual, predicted)
    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(actual, predicted)
    r2 = r2_score(actual, predicted)
    
    # Mean Absolute Percentage Error (alternative)
    mape_alt = np.mean(np.abs((actual - predicted) / actual)) * 100
    
    # Correlation
    correlation = np.corrcoef(actual, predicted)[0, 1]
    
    # Mean Percentage Error (bias)
    mpe = np.mean(((actual - predicted) / actual) * 100)
    
    # Normalized RMSE
    nrmse = rmse / np.mean(actual)
    
    # Theil's U statistic (relative to naive forecast)
    naive_forecast = actual.shift(1).dropna()
    actual_shifted = actual.iloc[1:]
    u_numerator = np.sqrt(np.mean((actual_shifted - predicted[:-1])**2))
    u_denominator = np.sqrt(np.mean(actual_shifted**2))
    theils_u = u_numerator / u_denominator if u_denominator != 0 else np.nan
    
    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'MAPE (%)': mape,
        'MAPE Alt (%)': mape_alt,
        'R²': r2,
        'Correlation': correlation,
        'MPE (%)': mpe,
        'NRMSE': nrmse,
        "Theil's U": theils_u
    }

metrics = calculate_all_metrics(test.values, predictions)

print("\n" + "="*70)
print("OPTIMIZED ARIMA MODEL - PERFORMANCE METRICS")
print("="*70)
for metric_name, metric_value in metrics.items():
    if isinstance(metric_value, float):
        if metric_name in ['MAPE (%)', 'MAPE Alt (%)', 'MPE (%)']:
            print(f"{metric_name:.<30} {metric_value:>10.2f}%")
        elif metric_name in ['R²', 'Correlation', 'NRMSE', "Theil's U"]:
            print(f"{metric_name:.<30} {metric_value:>10.4f}")
        else:
            print(f"{metric_name:.<30} {metric_value:>10.4f}")
    else:
        print(f"{metric_name:.<30} {metric_value:>10}")

print("="*70)

## 10. Visualization - Actual vs Predicted

In [ ]:
df_plot = pd.DataFrame({
    "DATA": test.index,
    "Actual": test.values,
    "Predicted": predictions
})

df_long = df_plot.melt(id_vars="DATA", value_vars=["Actual", "Predicted"], var_name="Serie", value_name="Cantitate")

fig = px.line(
    df_long,
    x="DATA",
    y="Cantitate",
    color="Serie",
    title=f"ARIMA ({selected_order}) – Actual vs Predicted",
    labels={"DATA": "Data", "Cantitate": "Cantitate vândută"}
)
fig.update_layout(
    xaxis_title="Data",
    yaxis_title="Cantitate vândută",
    hovermode="x unified",
    legend_title_text=""
)
fig.update_xaxes(rangeslider_visible=True)
fig.show()

# Residuals plot
residuals = test.values - predictions
fig2 = px.line(x=test.index, y=residuals, title="Residuals (Actual - Predicted)", labels={"x": "Data", "y": "Residual"})
fig2.update_layout(height=300)
fig2.show()

## 11. Save Results and Model

In [ ]:
# Save predictions and metrics
df_plot.to_csv("../figures/arima_94869_predictions.csv", index=False)

import joblib
joblib.dump(best_fitted, "../figures/arima_94869_model.joblib")

print("Results and model saved to ../figures/")